# Pokémon GO-inspired exploratory analysis

**Repository role:** first runnable notebook for the persistent TI2006B project repository  
**Version:** v1.0  
**Status:** example

**License:** Original notebook narrative and teaching material are licensed under [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/). The dataset source and software libraries retain their own terms.

## Purpose

This notebook demonstrates a small, reproducible analysis that can be reviewed in real time. We use public Pokémon species and base-stat tables to practice ingestion, joins, data quality checks, derived metrics, and interactive communication with Plotly. The context is Pokémon GO-inspired, but the data is **not** player telemetry, location history, battle history, or an official Niantic dataset.

The initial business-style question is: **How do Pokémon types and base statistics vary across the available records, and what can a simple visual analysis communicate to a product or analytics team?**

## Dataset and provenance

The source tables are downloaded at runtime from the public `veekun/pokedex` repository. The notebook intentionally avoids committing a duplicate dataset file. The source is community-maintained Pokédex data; verify its license, version, and freshness before making a public or commercial claim.

- `pokemon.csv`: identifiers, names, height, weight, and base experience.
- `pokemon_stats.csv`: one row per Pokémon/statistic pair and base-stat value.
- `pokemon_types.csv`: one row per Pokémon/type pair and type slot.
- `stats.csv` and `types.csv`: lookup tables for readable labels.

The source does not measure Pokémon GO encounters or player behavior. That distinction is part of the analysis evidence.

In [ ]:
from datetime import datetime, timezone
from io import StringIO
from urllib.request import urlopen

import pandas as pd
import plotly.express as px

pd.set_option("display.max_columns", 30)
RETRIEVED_AT = datetime.now(timezone.utc).isoformat(timespec="seconds")
BASE_URL = "https://raw.githubusercontent.com/veekun/pokedex/master/pokedex/data/csv/"

def read_source(filename: str) -> pd.DataFrame:
    with urlopen(BASE_URL + filename, timeout=30) as response:
        return pd.read_csv(StringIO(response.read().decode("utf-8")))

print(f"Retrieved at: {RETRIEVED_AT}")

In [ ]:
pokemon = read_source("pokemon.csv")
pokemon_stats = read_source("pokemon_stats.csv")
pokemon_types = read_source("pokemon_types.csv")
stats = read_source("stats.csv")
types = read_source("types.csv")

# Keep default forms for a compact first analysis.
pokemon = pokemon.loc[pokemon["is_default"] == 1].copy()
stats = stats[["id", "identifier"]].rename(columns={"id": "stat_id", "identifier": "stat_name"})
types = types[["id", "identifier"]].rename(columns={"id": "type_id", "identifier": "type_name"})

pokemon.head()

## Data-quality checks

Before visualizing, check row counts, duplicate identifiers, missing names, and whether every statistic and type can be mapped to a readable label. These checks are intentionally simple so later activities can turn them into automated tests.

In [ ]:
quality = {
    "pokemon_rows": len(pokemon),
    "duplicate_pokemon_ids": int(pokemon["id"].duplicated().sum()),
    "missing_identifiers": int(pokemon["identifier"].isna().sum()),
    "stat_rows": len(pokemon_stats),
    "type_rows": len(pokemon_types),
}
pd.Series(quality, name="value").to_frame()

In [ ]:
stats_tidy = (
    pokemon_stats.merge(stats, on="stat_id", how="left")
    .pivot_table(index="pokemon_id", columns="stat_name", values="base_stat", aggfunc="first")
    .reset_index()
)
stats_tidy["total_base_stats"] = stats_tidy.drop(columns="pokemon_id").sum(axis=1)

analysis = pokemon[["id", "identifier", "height", "weight", "base_experience"]].merge(
    stats_tidy, left_on="id", right_on="pokemon_id", how="left"
).drop(columns="pokemon_id")

types_tidy = pokemon_types.merge(types, on="type_id", how="left")
analysis = analysis.merge(
    types_tidy.groupby("pokemon_id")["type_name"].agg(lambda values: ", ".join(sorted(values))).rename("types"),
    left_on="id", right_index=True, how="left"
).rename(columns={"identifier": "name"})

analysis.head()

## Interactive visual analysis

The figures below are descriptive. A higher total base-stat value is not the same as Pokémon GO performance, battle ranking, or player value. The visualizations help us ask better questions; they do not answer questions that the dataset cannot measure.

In [ ]:
top = analysis.nlargest(20, "total_base_stats").sort_values("total_base_stats")
fig = px.bar(
    top,
    x="total_base_stats",
    y="name",
    color="types",
    orientation="h",
    title="Top 20 default-form Pokémon by total base stats",
    labels={"total_base_stats": "Total base stats", "name": "Pokémon", "types": "Types"},
)
fig.update_layout(template="plotly_white", height=650)
fig.show()

In [ ]:
type_counts = (
    types_tidy.merge(pokemon[["id", "is_default"]], left_on="pokemon_id", right_on="id")
    .query("is_default == 1")
    .groupby("type_name", as_index=False)["pokemon_id"]
    .nunique()
    .rename(columns={"pokemon_id": "pokemon_count"})
    .sort_values("pokemon_count", ascending=False)
)

fig = px.bar(
    type_counts,
    x="type_name",
    y="pokemon_count",
    color="pokemon_count",
    color_continuous_scale="Viridis",
    title="Default-form Pokémon by type",
    labels={"type_name": "Type", "pokemon_count": "Pokémon count"},
)
fig.update_layout(template="plotly_white", xaxis_tickangle=-45)
fig.show()

In [ ]:
fig = px.scatter(
    analysis,
    x="weight",
    y="total_base_stats",
    size="base_experience",
    color="types",
    hover_name="name",
    log_x=True,
    title="Weight and total base stats",
    labels={"weight": "Weight (hectograms, log scale)", "total_base_stats": "Total base stats", "types": "Types"},
)
fig.update_layout(template="plotly_white", height=600)
fig.show()

## Interpretation and next steps

Write three observations and one limitation after running the notebook. A strong next step is to compare this species-level reference table with a clearly licensed Pokémon GO battle or move dataset, documenting how the meaning of each metric changes. Other extensions for the class repository include a data dictionary, a dimensional model, automated quality tests, a small dashboard, and a Jira issue linked to the pull request.

**Academic and privacy note:** do not upload personal Pokémon GO exports, account identifiers, precise locations, or screenshots containing private information without an approved learning purpose and explicit authorization.